In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# 차단 및 제재 키워드 정의
forbidden_topics = {
    "cheating": ["답지", "정답 알려줘", "숙제 대신", "써줘", "베끼기"], # 부정행위 관련
    "distraction": ["롤", "게임", "유튜브", "아이돌", "웹툰", "웃긴"], # 학습 방해 요소
    "harmful": ["담배", "술", "폭력", "싸움", "바보"] # 유해 콘텐츠
}

In [3]:
from langchain.agents.middleware import before_agent

@before_agent(can_jump_to=["end"])
def education_guardrail(state, runtime):
    """
    학생의 질문 의도를 파악하여 교육적이지 않거나 부정행위가 의심될 경우,
    LLM(AI)에게 질문을 넘기지 않고 교육적인 멘트로 즉시 교정합니다.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]:
        return None

    last_message = state["messages"][-1]
    if last_message.type != "human":
        return None

    user_text = last_message.content

    # 2. 카테고리별 검사 로직
    # 단순히 막는 것을 넘어, '왜' 안되는지 카테고리별로 다른 피드백 주기

    # Case A: 부정행위 방지 (Cheating Prevention)
    # AI가 숙제를 통째로 해주는 것을 방지
    for keyword in forbidden_topics["cheating"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요."
                }],
                "jump_to": "end"
            }

    # Case B: 학습 집중 유도 (Focus Management)
    # 공부 중에 게임이나 딴짓 이야기를 하면 다시 공부로 유도
    for keyword in forbidden_topics["distraction"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?"
                }],
                "jump_to": "end"
            }

    # Case C: 유해 콘텐츠 차단 (Safety)
    # 교육 서비스의 브랜드 안전성(Brand Safety)을 위한 기능
    for keyword in forbidden_topics["harmful"]:
        if keyword in user_text:
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요."
                }],
                "jump_to": "end"
            }

    # 3. 통과 (Pass)
    # 위 조건들에 걸리지 않으면 정상적으로 AI 튜터(LLM)가 답변 생성
    return None

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[education_guardrail],
)

In [5]:

agent.invoke({
    "messages": [{"role": "user", "content": "피타고라스의 정리가 이해가 안 돼. 설명해줘."}]
})

{'messages': [HumanMessage(content='피타고라스의 정리가 이해가 안 돼. 설명해줘.', additional_kwargs={}, response_metadata={}, id='18156468-0022-4515-9061-8e84fbcf4812'),
  AIMessage(content='피타고라스의 정리가 어렵게 느껴지시는군요! 걱정 마세요. 최대한 쉽고 명확하게 설명해 드릴게요.\n\n**핵심은 "직각삼각형"과 "넓이"입니다.**\n\n**1. 직각삼각형이란?**\n\n먼저, 피타고라스의 정리가 적용되는 특별한 삼각형이 있습니다. 바로 **직각삼각형**이에요. 직각삼각형은 이름 그대로 **한 각이 90도인 삼각형**을 말합니다.\n\n*   **빗변 (hypotenuse):** 직각삼각형에서 가장 긴 변이며, 직각과 마주보는 변을 말합니다.\n*   **다른 두 변 (legs):** 직각을 끼고 있는 두 개의 짧은 변들을 말합니다.\n\n**2. 피타고라스의 정리, 무엇을 말하는가?**\n\n피타고라스의 정리는 직각삼각형의 세 변의 길이 사이에 아주 특별한 관계가 있다는 것을 설명해 줍니다.\n\n**"직각삼각형에서 빗변의 길이를 제곱한 것은 다른 두 변의 길이를 각각 제곱하여 더한 것과 같다."**\n\n이것을 수학 공식으로 나타내면 다음과 같습니다.\n\n만약 직각삼각형의 다른 두 변의 길이를 각각 $a$와 $b$라고 하고, 빗변의 길이를 $c$라고 한다면,\n\n$a^2 + b^2 = c^2$\n\n이 공식이 바로 피타고라스의 정리입니다.\n\n**3. 왜 "제곱"일까요? 넓이로 이해하기!**\n\n"제곱"이라는 말이 조금 어렵게 느껴질 수 있습니다. 이것을 **넓이**로 생각하면 훨씬 이해하기 쉬워요.\n\n각 변을 한 변으로 하는 정사각형을 그려봅시다.\n\n*   변 $a$를 한 변으로 하는 정사각형의 넓이 = $a \\times a = a^2$\n*   변 $b$를 한 변으로 하는 정사각형의 넓이 = $b \\times b 

In [6]:

agent.invoke({
    "messages": [{"role": "user", "content": "독후감 대신 써줘."}]
})

{'messages': [HumanMessage(content='독후감 대신 써줘.', additional_kwargs={}, response_metadata={}, id='464346f9-9df9-4538-ab35-4736193aeeb1'),
  AIMessage(content='🚫 스스로 고민해봐야 실력이 늘어요! 정답을 바로 알려드리는 대신, 힌트를 드릴까요? 어떤 부분이 가장 어려운지 말해주세요.', additional_kwargs={}, response_metadata={}, id='907bf3f9-b72e-4fec-9bb5-cad5e7ddbf38', tool_calls=[], invalid_tool_calls=[])]}

In [7]:
agent.invoke({
    "messages": [{"role": "user", "content": "웃긴 얘기해줘"}]
})

{'messages': [HumanMessage(content='웃긴 얘기해줘', additional_kwargs={}, response_metadata={}, id='aabc9ab0-010b-4570-bfe3-ed3c8cdbb1de'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='c1e9dae7-461b-43d5-9fb9-f2624ca92178', tool_calls=[], invalid_tool_calls=[])]}

In [8]:
agent.invoke({
    "messages": [{"role": "user", "content": "재밌는 유튜브 알려줘"}]
})

{'messages': [HumanMessage(content='재밌는 유튜브 알려줘', additional_kwargs={}, response_metadata={}, id='54d06552-b77c-4227-8dd5-a9c87f81eec1'),
  AIMessage(content='⏰ 지금은 공부에 집중할 시간이에요! 딴짓은 쉬는 시간에 하고, 지금 풀고 있는 문제에 집중해볼까요?', additional_kwargs={}, response_metadata={}, id='f37fc879-cb56-4fac-a0fb-8f81b10250db', tool_calls=[], invalid_tool_calls=[])]}

In [9]:
agent.invoke({
    "messages": [{"role": "user", "content": "담배 피면 좋아?"}]
})

{'messages': [HumanMessage(content='담배 피면 좋아?', additional_kwargs={}, response_metadata={}, id='6437dadc-9ad0-4e92-95f4-1a43d6215ffe'),
  AIMessage(content='⚠️ 부적절한 대화 주제입니다. 바르고 고운 말을 사용해주세요.', additional_kwargs={}, response_metadata={}, id='36ec2344-da1a-4551-96b2-ccb52f84c3f0', tool_calls=[], invalid_tool_calls=[])]}

In [10]:
from langchain.chat_models import init_chat_model

safety_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [11]:
from langchain.agents.middleware import after_agent
from langchain.messages import AIMessage

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은..."

    return None

In [12]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[answer_leakage_guardrail],
)

In [13]:
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘."}]
})

🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='3cd0c275-5b38-48cf-be01-1ec443f67ece'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은...', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d80de-7945-7222-a0ec-8615ad76ff10-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 179, 'total_tokens': 207, 'input_token_details': {'cache_read': 0}})]}

In [14]:
import re

@before_agent
def student_safety_middleware(state, runtime):
    """
    학생의 전화번호나 이메일이 감지되면 마스킹 처리하여 안전을 확보
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]
    if last_message.type != "human": return None

    content = last_message.content
    original_content = content # 로깅용

    # 전화번호 패턴 (010-XXXX-XXXX 또는 010XXXXXXXX 등)
    phone_pattern = r'01[016789]-?[0-9]{3,4}-?[0-9]{4}'
    # 이메일 패턴
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

    is_redacted = False

    if re.search(phone_pattern, content):
        content = re.sub(phone_pattern, '<PHONE_REDACTED>', content)
        is_redacted = True

    if re.search(email_pattern, content):
        content = re.sub(email_pattern, '<EMAIL_REDACTED>', content)
        is_redacted = True

    if is_redacted:
        print(f"🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.\n원본: {original_content}\n수정: {content}")
        # 내용을 수정하여 LLM에게 전달 (사용자에게 알릴 필요 없이 조용히 처리하거나, 시스템 메시지 추가 가능)
        last_message.content = content

    return None

In [15]:
ESCALATION_KEYWORDS = ["왕따", "괴롭힘", "우울해", "학교 폭력", "상담 선생님", "사람 불러줘"]

@before_agent(can_jump_to=["end"])
def counseling_escalation_middleware(state, runtime) :
    """
    [Layer 3] 심리적 위기 상황이나 상담 요청이 감지되면 AI 답변을 멈추고 인간 상담사에게 알림을 보냅니다.
    """
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 민감한 키워드가 포함되어 있는지 확인
    for keyword in ESCALATION_KEYWORDS:
        if keyword in last_message.content:
            print(f"✋ [상담 이관] 심각한 고민/요청 감지: {keyword}")

            # 여기서 실제로는 상담 교사에게 알림(Slack, Email 등)을 보내는 로직이 들어감
            # send_alert_to_teacher(last_message.content)

            return {
                "messages": [{
                    "role": "assistant",
                    "content": "학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)"
                }],
                "jump_to": "end" # AI 답변 생성 중단
            }
    return None

In [16]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[
        education_guardrail,             # Layer 1: 입력 필터 (규칙 - 딴짓/부정행위)
        student_safety_middleware,       # Layer 2: 개인정보 보호 (전화번호 마스킹)
        counseling_escalation_middleware,# Layer 3: 상담 이관 (휴먼 에스컬레이션)
        answer_leakage_guardrail         # Layer 4: 출력 필터 (모델 기반 교정)
    ],
)

In [17]:
agent.invoke({
    "messages": [{"role": "user", "content": "제 번호 010-1234-5678입니다."}]
})

🔒 [학생 보호] 개인정보가 감지되어 마스킹 처리했습니다.
원본: 제 번호 010-1234-5678입니다.
수정: 제 번호 <PHONE_REDACTED>입니다.


{'messages': [HumanMessage(content='제 번호 <PHONE_REDACTED>입니다.', additional_kwargs={}, response_metadata={}, id='7235b098-ca40-4007-985e-e226bde5d3ba'),
  AIMessage(content='네, <PHONE_REDACTED>번으로 확인되었습니다. 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d80e2-3df4-7c52-a30b-e73bb82a9ffe-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 20, 'total_tokens': 33, 'input_token_details': {'cache_read': 0}})]}

In [18]:
agent.invoke({
    "messages": [{"role": "user", "content": "요즘 학교에서 왕따 당하고 있어"}]
})

✋ [상담 이관] 심각한 고민/요청 감지: 왕따


{'messages': [HumanMessage(content='요즘 학교에서 왕따 당하고 있어', additional_kwargs={}, response_metadata={}, id='0f58c066-bfee-4d27-9ebd-8f03eb8a167a'),
  AIMessage(content='학생, 많이 힘들었겠구나. 이 문제는 내가 답변하기보다는 전문 상담 선생님이 직접 듣고 도와주시는 게 좋을 것 같아. \n\n지금 바로 상담 선생님께 연결해 드렸으니 잠시만 기다려 줄래? 🍀 (상담실 연결 중...)', additional_kwargs={}, response_metadata={}, id='47438b73-0d3b-4a82-af09-af1f53caaece', tool_calls=[], invalid_tool_calls=[])]}